# PalanthAI — BGE-M3 Embedding

**GitHub → Clone → Embed → Drive Upload (user OAuth) → n8n**


In [ ]:
# 1. Clone data from GitHub
!git clone https://github.com/jeanhackpy/palanthai-colab.git /content/palanthai-colab
%cd /content/palanthai-colab
!pip install -q FlagEmbedding pyarrow tqdm pandas 2>&1 | tail -2
print('Setup OK — repo clone done')

In [ ]:
# 2. Authenticate Drive for upload (user OAuth — NOT service account)
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive', timeout_secs=300)

from google.auth import default
creds, _ = default()
from googleapiclient.discovery import build
drive_api = build('drive', 'v3', credentials=creds)

FOLDER_ID = '13Mc5Nu2mgtQcsOFuN79Ug6-i1mRFPE9q'
N8N_WEBHOOK = 'https://n8n.recall-agency.com/webhook/palanthai-colab-download'

DATA_DIR = Path('/content/palanthai-colab')
CSV_FILE = DATA_DIR / 'project_embeddings_20260331_5k_part0.csv'
print(f'CSV: {CSV_FILE} ({CSV_FILE.stat().st_size // 1024} KB)')

In [ ]:
# 3. Load BGE-M3
import torch
print('GPU:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print('Model OK')

In [ ]:
# 4. Embed + save parquet locally
from tqdm.notebook import tqdm
import pandas as pd

df = pd.read_csv(CSV_FILE)
ids   = df['project_id'].tolist()
texts = df['text'].fillna('').tolist()
print(f'Embedding {len(texts)} projets...')

out = model.encode(texts, batch_size=64, max_length=512,
                    return_dense=True, return_sparse=True)

res = pd.DataFrame({
    'project_id':    ids,
    'dense_vector':  out['dense_vecs'].tolist(),
    'sparse_indices': [list(s.keys()) for s in out['lexical_weights']],
    'sparse_values':  [list(s.values()) for s in out['lexical_weights']],
})
for col in ['region']:
    if col in df.columns: res[col] = df[col]

pq_name = 'project_embeddings_20260331_5k_part0_vectors.parquet'
local_pq = Path('/content/') / pq_name
res.to_parquet(local_pq, index=False)
print(f'Sauvegarde: {pq_name} ({local_pq.stat().st_size // 1024} KB)')
del df, res, out
print('Embedding TERMINE')

In [ ]:
# 5. Upload parquet to Drive (user OAuth — works!)
from googleapiclient.http import MediaFileUpload

media = MediaFileUpload(str(local_pq),
    mimetype='application/octet-stream', resumable=True)

# Check if already exists
query = f"name='{pq_name}' and '{FOLDER_ID}' in parents and trashed=false"
existing = drive_api.files().list(
    q=query, fields='files(id)').execute()

if existing.get('files'):
    fid = existing['files'][0]['id']
    drive_api.files().update(
        fileId=fid, media_body=media, supportsAllDrives=True).execute()
    print(f'Updated: {pq_name}')
else:
    drive_api.files().create(
        body={'name': pq_name, 'parents': [FOLDER_ID]},
        media_body=media, fields='id', supportsAllDrives=True).execute()
    print(f'Uploaded: {pq_name}')

print('Drive upload OK')

In [ ]:
# 6. Notify n8n
import requests
payload = {
    'status': 'parquet_uploaded',
    'parquet_files': [pq_name],
    'collection': 'projects_v3',
    'count': 1,
    'source': 'colab'
}
try:
    r = requests.post(N8N_WEBHOOK, json=payload, timeout=30)
    print(f'n8n: {r.status_code} - {r.text[:200]}')
except Exception as e:
    print(f'n8n failed: {e}')

print('DONE — ', pq_name)